# 3D Aorta Segmentation — run on Google Colab

Runtime → Change runtime type → **GPU**. Then run the cells in order.

## 1. Get the code and install requirements

In [ ]:
!git clone https://github.com/<your-username>/aorta-seg-3d.git
%cd aorta-seg-3d
!pip install -q -r requirements.txt

## 2. Point the config to your data on Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import yaml
cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['data']['volumes_dir'] = '/content/drive/MyDrive/CIS_UNet_Dataset/Volumes'
cfg['data']['labels_dir']  = '/content/drive/MyDrive/CIS_UNet_Dataset/Labels'
cfg['output_dir'] = '/content/drive/MyDrive/aorta_outputs'
yaml.safe_dump(cfg, open('configs/colab.yaml', 'w'))
print(cfg)

## 3. Train and evaluate

In [ ]:
!python -m aortaseg.train --config configs/colab.yaml

In [ ]:
!python -m aortaseg.evaluate --config configs/colab.yaml --save-predictions
!cat /content/drive/MyDrive/aorta_outputs/test_summary.json

## 4. Training curve and an example prediction for the README

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk

out = cfg['output_dir']
h = json.load(open(os.path.join(out, 'history.json')))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot([r['train_loss'] for r in h]); ax[0].set_title('Training loss'); ax[0].set_xlabel('Epoch')
ax[1].plot([r['val_dice'] for r in h]); ax[1].set_title('Validation Dice'); ax[1].set_xlabel('Epoch')
plt.tight_layout(); os.makedirs('results', exist_ok=True); plt.savefig('results/training_curves.png', dpi=150); plt.show()

vol_path, lab_path = json.load(open(os.path.join(out, 'splits.json')))['test'][0]
vol = sitk.GetArrayFromImage(sitk.ReadImage(vol_path))
gt = sitk.GetArrayFromImage(sitk.ReadImage(lab_path)) > 0
pred = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(out, 'predictions', os.path.basename(vol_path)))) > 0
z = int(np.argmax(gt.sum(axis=(1, 2))))          # slice with the most aorta
img = np.clip(vol[z], -200, 800)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (title, m) in zip(ax, [('CTA', None), ('Ground truth', gt[z]), ('Prediction', pred[z])]):
    a.imshow(img, cmap='gray')
    if m is not None: a.imshow(np.ma.masked_where(~m, m), cmap='autumn', alpha=0.6)
    a.set_title(title); a.axis('off')
plt.tight_layout(); plt.savefig('results/example_prediction.png', dpi=150, bbox_inches='tight'); plt.show()

Download `results/example_prediction.png` and `results/training_curves.png`, add them to the repository's `results/` folder, and copy the numbers from `test_summary.json` into the README table.